In [4]:
# importing libraries
import os
import cv2
from tqdm import tqdm

In [5]:
# configuring path to the files
ROOT = "dataset_split"
SPLITS = [
    "train",
    "test"
]

In [6]:
# for each train and test spli, it forms the folder degraded probes
for split in SPLITS:
    os.makedirs(
        os.path.join(
            ROOT,
            split,
            "degraded_probes"
        ),
        exist_ok=True
    )

print("Folders created")

Folders created


In [7]:
# degradation functions
def blur_low(img):
    return cv2.GaussianBlur(
        img,
        (5,5),
        0
    )

def blur_high(img):
    return cv2.GaussianBlur(
        img,
        (11,11),
        0
    )

def jpeg_low(img):
    encode_param = [
        int(cv2.IMWRITE_JPEG_QUALITY),
        50
    ]
    _, enc = cv2.imencode(
        ".jpg",
        img,
        encode_param
    )
    return cv2.imdecode(
        enc,
        1
    )

def jpeg_high(img):
    encode_param = [
        int(cv2.IMWRITE_JPEG_QUALITY),
        15
    ]
    _, enc = cv2.imencode(
        ".jpg",
        img,
        encode_param
    )
    return cv2.imdecode(
        enc,
        1
    )

In [8]:
# processing training and tesing sets
for split in SPLITS:
    input_root = os.path.join(
        ROOT,
        split,
        "probe_cropped"
    )
    output_root = os.path.join(
        ROOT,
        split,
        "degraded_probes"
    )
    identities = os.listdir(
        input_root
    )
    print(
        f"\nProcessing {split}"
    )
    for identity in tqdm(
        identities
    ):
        src_dir = os.path.join(
            input_root,
            identity
        )
        dst_dir = os.path.join(
            output_root,
            identity
        )
        os.makedirs(
            dst_dir,
            exist_ok=True
        )
        images = os.listdir(
            src_dir
        )
        for image_name in images:
            image_path = os.path.join(
                src_dir,
                image_name
            )
            img = cv2.imread(
                image_path
            )
            if img is None:
                continue
            base = os.path.splitext(
                image_name
            )[0]
            degradations = {
                "blur_low":
                    blur_low(img),
                "blur_high":
                    blur_high(img),
                "jpeg_low":
                    jpeg_low(img),
                "jpeg_high":
                    jpeg_high(img)
            }
            for deg_name, deg_img in degradations.items():
                save_path = os.path.join(
                    dst_dir,
                    f"{base}_{deg_name}.jpg"
                )
                cv2.imwrite(
                    save_path,
                    deg_img
                )
print("\nDegradation complete.")


Processing train


100%|██████████| 100/100 [00:00<00:00, 127.27it/s]



Processing test


100%|██████████| 30/30 [00:00<00:00, 159.19it/s]


Degradation complete.


In [9]:
# verification for the number of probe images
for split in SPLITS:
    total = 0
    root = os.path.join(
        ROOT,
        split,
        "degraded_probes"
    )
    for identity in os.listdir(root):

        total += len(
            os.listdir(
                os.path.join(
                    root,
                    identity
                )
            )
        )
    print(
        split,
        total
    )

train 2196
test 540
